In [1]:
# Проверка окружения
import sys
import sqlite3
import os

print("=== Проверка окружения ===")
print(f"Python: {sys.version}")
print(f"SQLite: {sqlite3.sqlite_version}")
print(f"Current directory: {os.getcwd()}")

# Создаем папку для данных
db_dir = "../data"
os.makedirs(db_dir, exist_ok=True)
print(f"Database directory: {os.path.abspath(db_dir)}")

# Проверка библиотек
print("\n=== Проверка библиотек ===")

try:
    import langgraph
    print(" LangGraph: установлен")
except ImportError:
    print(" LangGraph: НЕ УСТАНОВЛЕН")

try:
    from langchain_core.tools import tool
    print(" LangChain-core: установлен")
except ImportError:
    print(" LangChain-core: НЕ УСТАНОВЛЕН")

try:
    import openai
    print(f" OpenAI: {openai.__version__}")
except ImportError:
    print(" OpenAI: НЕ УСТАНОВЛЕН")

try:
    from dotenv import load_dotenv
    print(" python-dotenv: установлен")
except ImportError:
    print(" python-dotenv: НЕ УСТАНОВЛЕН")

print("\n Готов к работе!")

=== Проверка окружения ===
Python: 3.11.6 | packaged by conda-forge | (main, Oct  3 2023, 10:40:35) [GCC 12.3.0]
SQLite: 3.43.2
Current directory: /home/jovyan/work/projects/recipe-agent/notebooks
Database directory: /home/jovyan/work/projects/recipe-agent/data

=== Проверка библиотек ===
 LangGraph: установлен
 LangChain-core: установлен
 OpenAI: 2.30.0
 python-dotenv: установлен

 Готов к работе!


In [2]:
import sqlite3
import os
from pathlib import Path

class RecipeDB:
    def __init__(self, db_path="../data/recipes.db"):
        """
        Инициализация базы данных.
        
        Args:
            db_path: путь к файлу базы данных SQLite
        """
        self.db_path = db_path
        self._init_tables()
        print(f" База данных создана: {db_path}")
    
    def _init_tables(self):
        """Создание всех таблиц, если они не существуют."""
        with sqlite3.connect(self.db_path) as conn:
            # 1. Таблица коллекций
            conn.execute("""
                CREATE TABLE IF NOT EXISTS collections (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    name TEXT UNIQUE NOT NULL,
                    description TEXT
                )
            """)
            print("   Таблица 'collections' создана")
            
            # 2. Таблица рецептов
            conn.execute("""
                CREATE TABLE IF NOT EXISTS recipes (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    collection_id INTEGER NOT NULL,
                    name TEXT NOT NULL,
                    description TEXT,
                    servings INTEGER,
                    prep_time INTEGER,
                    FOREIGN KEY (collection_id) REFERENCES collections(id) ON DELETE CASCADE
                )
            """)
            print("   Таблица 'recipes' создана")
            
            # 3. Таблица ингредиентов
            conn.execute("""
                CREATE TABLE IF NOT EXISTS ingredients (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    recipe_id INTEGER NOT NULL,
                    name TEXT NOT NULL,
                    amount REAL,
                    unit TEXT,
                    FOREIGN KEY (recipe_id) REFERENCES recipes(id) ON DELETE CASCADE
                )
            """)
            print("   Таблица 'ingredients' создана")
            
            # 4. Таблица шагов приготовления
            conn.execute("""
                CREATE TABLE IF NOT EXISTS steps (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    recipe_id INTEGER NOT NULL,
                    step_order INTEGER NOT NULL,
                    description TEXT NOT NULL,
                    FOREIGN KEY (recipe_id) REFERENCES recipes(id) ON DELETE CASCADE
                )
            """)
            print("   Таблица 'steps' создана")

# Создаем экземпляр базы данных
db = RecipeDB()

   Таблица 'collections' создана
   Таблица 'recipes' создана
   Таблица 'ingredients' создана
   Таблица 'steps' создана
 База данных создана: ../data/recipes.db


In [20]:
# Создаем коллекцию, если её нет
if not db.collection_exists("Italian Cuisine"):
    db.create_collection("Italian Cuisine", "Classic Italian recipes")
    print(" Коллекция 'Italian Cuisine' создана")
else:
    print(" Коллекция 'Italian Cuisine' уже существует")

# Получаем коллекцию для дальнейшего использования
italian = db.get_collection_by_name("Italian Cuisine")
print(f"ID коллекции: {italian['id']}")

 Коллекция 'Italian Cuisine' уже существует
ID коллекции: 1


In [17]:
class RecipeDB(RecipeDB):  # Расширяем класс
    def __init__(self, db_path="../data/recipes.db"):
        """
        Инициализация базы данных.
        
        Args:
            db_path: путь к файлу базы данных SQLite
        """
        self.db_path = db_path
        self._init_tables()
        print(f" База данных создана: {db_path}")
    
    def _init_tables(self):
        """Создание всех таблиц, если они не существуют."""
        with sqlite3.connect(self.db_path) as conn:
            # 1. Таблица коллекций
            conn.execute("""
                CREATE TABLE IF NOT EXISTS collections (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    name TEXT UNIQUE NOT NULL,
                    description TEXT
                )
            """)
            print("   Таблица 'collections' создана")
            
            # 2. Таблица рецептов
            conn.execute("""
                CREATE TABLE IF NOT EXISTS recipes (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    collection_id INTEGER NOT NULL,
                    name TEXT NOT NULL,
                    description TEXT,
                    servings INTEGER,
                    prep_time INTEGER,
                    FOREIGN KEY (collection_id) REFERENCES collections(id) ON DELETE CASCADE
                )
            """)
            print("   Таблица 'recipes' создана")
            
            # 3. Таблица ингредиентов
            conn.execute("""
                CREATE TABLE IF NOT EXISTS ingredients (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    recipe_id INTEGER NOT NULL,
                    name TEXT NOT NULL,
                    amount REAL,
                    unit TEXT,
                    FOREIGN KEY (recipe_id) REFERENCES recipes(id) ON DELETE CASCADE
                )
            """)
            print("   Таблица 'ingredients' создана")
            
            # 4. Таблица шагов приготовления
            conn.execute("""
                CREATE TABLE IF NOT EXISTS steps (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    recipe_id INTEGER NOT NULL,
                    step_order INTEGER NOT NULL,
                    description TEXT NOT NULL,
                    FOREIGN KEY (recipe_id) REFERENCES recipes(id) ON DELETE CASCADE
                )
            """)
            print("   Таблица 'steps' создана")    
    
    
    
    # Методы для коллекций
    
    def create_collection(self, name: str, description: str = "") -> int:
        """
        Создать новую коллекцию.
        
        Args:
            name: название коллекции
            description: описание (опционально)
            
        Returns:
            id созданной коллекции или None при ошибке
        """
        with sqlite3.connect(self.db_path) as conn:
            try:
                cursor = conn.execute(
                    "INSERT INTO collections (name, description) VALUES (?, ?)",
                    (name, description)
                )
                conn.commit()
                print(f" Коллекция '{name}' создана (ID: {cursor.lastrowid})")
                return cursor.lastrowid
            except sqlite3.IntegrityError:
                print(f" Коллекция '{name}' уже существует")
                return None
    
    def get_collection(self, collection_id: int):
        """
        Получить коллекцию по ID.
        
        Args:
            collection_id: id коллекции
            
        Returns:
            словарь с данными коллекции или None
        """
        with sqlite3.connect(self.db_path) as conn:
            conn.row_factory = sqlite3.Row
            cursor = conn.execute(
                "SELECT * FROM collections WHERE id = ?",
                (collection_id,)
            )
            row = cursor.fetchone()
            return dict(row) if row else None
    
    def get_collection_by_name(self, name: str):
        """Получить коллекцию по названию."""
        with sqlite3.connect(self.db_path) as conn:
            conn.row_factory = sqlite3.Row
            cursor = conn.execute(
                "SELECT * FROM collections WHERE name = ?",
                (name,)
            )
            row = cursor.fetchone()
            return dict(row) if row else None
    
    def list_collections(self) -> list:
        """Получить список всех коллекций."""
        with sqlite3.connect(self.db_path) as conn:
            conn.row_factory = sqlite3.Row
            cursor = conn.execute("SELECT * FROM collections ORDER BY name")
            return [dict(row) for row in cursor.fetchall()]
    def create_recipe(self, collection_id: int, name: str, 
                      servings: int = None, prep_time: int = None,
                      description: str = "") -> int:
        """
        Создать новый рецепт.
        
        Args:
            collection_id: id коллекции
            name: название рецепта
            servings: количество порций
            prep_time: время приготовления (минуты)
            description: описание
            
        Returns:
            id созданного рецепта
        """
        with sqlite3.connect(self.db_path) as conn:
            cursor = conn.execute(
                """INSERT INTO recipes 
                   (collection_id, name, description, servings, prep_time)
                   VALUES (?, ?, ?, ?, ?)""",
                (collection_id, name, description, servings, prep_time)
            )
            conn.commit()
            print(f" Рецепт '{name}' создан (ID: {cursor.lastrowid})")
            return cursor.lastrowid
    
    def get_recipe(self, recipe_id: int):
        """Получить рецепт по ID."""
        with sqlite3.connect(self.db_path) as conn:
            conn.row_factory = sqlite3.Row
            cursor = conn.execute(
                "SELECT * FROM recipes WHERE id = ?", (recipe_id,)
            )
            row = cursor.fetchone()
            return dict(row) if row else None
    
    def list_recipes(self, collection_id: int) -> list:
        """Получить все рецепты в коллекции."""
        with sqlite3.connect(self.db_path) as conn:
            conn.row_factory = sqlite3.Row
            cursor = conn.execute(
                "SELECT * FROM recipes WHERE collection_id = ? ORDER BY name",
                (collection_id,)
            )
            return [dict(row) for row in cursor.fetchall()]
    def add_ingredient(self, recipe_id: int, name: str, 
                       amount: float = None, unit: str = None) -> int:
        """
        Добавить ингредиент в рецепт.
        
        Args:
            recipe_id: id рецепта
            name: название ингредиента
            amount: количество
            unit: единица измерения (g, kg, ml, l, pcs, tbsp, tsp)
            
        Returns:
            id добавленного ингредиента
        """
        with sqlite3.connect(self.db_path) as conn:
            cursor = conn.execute(
                "INSERT INTO ingredients (recipe_id, name, amount, unit) VALUES (?, ?, ?, ?)",
                (recipe_id, name, amount, unit)
            )
            conn.commit()
            print(f"   Ингредиент '{name}' добавлен")
            return cursor.lastrowid
    
    def get_ingredients(self, recipe_id: int) -> list:
        """Получить все ингредиенты рецепта."""
        with sqlite3.connect(self.db_path) as conn:
            conn.row_factory = sqlite3.Row
            cursor = conn.execute(
                "SELECT * FROM ingredients WHERE recipe_id = ?",
                (recipe_id,)
            )
            return [dict(row) for row in cursor.fetchall()]
    def add_step(self, recipe_id: int, step_order: int, description: str) -> int:
        """
        Добавить шаг приготовления.
        
        Args:
            recipe_id: id рецепта
            step_order: номер шага (1, 2, 3...)
            description: описание шага
            
        Returns:
            id добавленного шага
        """
        with sqlite3.connect(self.db_path) as conn:
            cursor = conn.execute(
                "INSERT INTO steps (recipe_id, step_order, description) VALUES (?, ?, ?)",
                (recipe_id, step_order, description)
            )
            conn.commit()
            print(f"   Шаг {step_order} добавлен")
            return cursor.lastrowid
    
    def get_steps(self, recipe_id: int) -> list:
        """Получить все шаги рецепта (в порядке возрастания)."""
        with sqlite3.connect(self.db_path) as conn:
            conn.row_factory = sqlite3.Row
            cursor = conn.execute(
                "SELECT * FROM steps WHERE recipe_id = ? ORDER BY step_order",
                (recipe_id,)
            )
            return [dict(row) for row in cursor.fetchall()]
    
    def update_step(self, recipe_id: int, step_order: int, new_description: str) -> bool:
        """
        Обновить шаг по его номеру.
        
        Args:
            recipe_id: id рецепта
            step_order: номер шага
            new_description: новое описание
            
        Returns:
            True если шаг обновлен, False если не найден
        """
        with sqlite3.connect(self.db_path) as conn:
            cursor = conn.execute(
                "UPDATE steps SET description = ? WHERE recipe_id = ? AND step_order = ?",
                (new_description, recipe_id, step_order)
            )
            conn.commit()
            if cursor.rowcount > 0:
                print(f"   Шаг {step_order} обновлен")
                return True
            else:
                print(f"   Шаг {step_order} не найден")
                return False
    def get_full_recipe(self, recipe_id: int) -> dict:
        """
        Получить полный рецепт с ингредиентами и шагами.
        
        Args:
            recipe_id: id рецепта
            
        Returns:
            словарь с рецептом, ингредиентами и шагами
        """
        recipe = self.get_recipe(recipe_id)
        if not recipe:
            return None
        
        recipe['ingredients'] = self.get_ingredients(recipe_id)
        recipe['steps'] = self.get_steps(recipe_id)
        
        return recipe
        
    def collection_exists(self, name: str) -> bool:
        """Проверить, существует ли коллекция с таким названием."""
        with sqlite3.connect(self.db_path) as conn:
            cursor = conn.execute(
                "SELECT id FROM collections WHERE name = ?", (name,)
            )
            return cursor.fetchone() is not None
    
    def recipe_exists(self, collection_id: int, name: str) -> bool:
        """Проверить, существует ли рецепт с таким названием в коллекции."""
        with sqlite3.connect(self.db_path) as conn:
            cursor = conn.execute(
                "SELECT id FROM recipes WHERE collection_id = ? AND name = ?",
                (collection_id, name)
            )
            return cursor.fetchone() is not None
    
    def get_recipe_by_name(self, collection_id: int, name: str):
        """Получить рецепт по названию и коллекции."""
        with sqlite3.connect(self.db_path) as conn:
            conn.row_factory = sqlite3.Row
            cursor = conn.execute(
                "SELECT * FROM recipes WHERE collection_id = ? AND name = ?",
                (collection_id, name)
            )
            row = cursor.fetchone()
            return dict(row) if row else None
    def show_all_recipes(db):
        """Показать все рецепты в базе данных."""
        with sqlite3.connect(db.db_path) as conn:
            conn.row_factory = sqlite3.Row
            cursor = conn.execute("""
                SELECT r.id, r.name, 
                       COUNT(DISTINCT i.id) as ingredients_count,
                       COUNT(DISTINCT s.id) as steps_count
                FROM recipes r
                LEFT JOIN ingredients i ON r.id = i.recipe_id
                LEFT JOIN steps s ON r.id = s.recipe_id
                GROUP BY r.id
                ORDER BY r.name
            """)
            recipes = cursor.fetchall()
            
            if not recipes:
                print(" No recipes found.")
                return
            
            print("\n ALL RECIPES:")
    
            for recipe in recipes:
                print(f"  {recipe['id']}. {recipe['name']}")
                print(f"     Ingredients: {recipe['ingredients_count']}")
                print(f"     Steps: {recipe['steps_count']}")
                print()


In [21]:
# Проверяем все рецепты в базе
print("  Все рецепты в БД")
with sqlite3.connect(db.db_path) as conn:
    conn.row_factory = sqlite3.Row
    cursor = conn.execute("SELECT id, name FROM recipes")
    recipes = cursor.fetchall()
    
    if not recipes:
        print("   Нет рецептов!")
    else:
        for recipe in recipes:
            print(f"  ID: {recipe['id']} | Название: {recipe['name']}")

  Все рецепты в БД
  ID: 1 | Название: Carbonara
  ID: 7 | Название: Margherita Pizza
  ID: 9 | Название: Tiramisu
  ID: 10 | Название: Creme Brulee
  ID: 11 | Название: Croissant


In [22]:
db = RecipeDB()
if not db.recipe_exists(italian['id'], "Carbonara"):
    recipe_id = db.create_recipe(italian['id'], "Carbonara", 2, 30, "Classic Roman pasta dish")
    db.add_ingredient(recipe_id, "spaghetti", 400, "g")
    db.add_ingredient(recipe_id, "guanciale", 200, "g")
    db.add_ingredient(recipe_id, "eggs", 4, "pcs")
    db.add_ingredient(recipe_id, "pecorino cheese", 100, "g")
    db.add_ingredient(recipe_id, "black pepper", 5, "g")
    db.add_step(recipe_id, 1, "Bring a large pot of salted water to boil")
    db.add_step(recipe_id, 2, "Cook spaghetti according to package instructions")
    db.add_step(recipe_id, 3, "Fry guanciale until golden and crispy")
    db.add_step(recipe_id, 4, "Mix eggs with pecorino cheese and pepper")
    db.add_step(recipe_id, 5, "Combine pasta with guanciale, then add egg mixture")
    db.add_step(recipe_id, 6, "Stir well, serve immediately")
    print(" Created recipe 'Carbonara'")
if not db.recipe_exists(italian['id'], "Margherita Pizza"):
    recipe_id = db.create_recipe(italian['id'], "Margherita Pizza", 1, 20, "Classic Neapolitan pizza")
    db.add_ingredient(recipe_id, "pizza dough", 250, "g")
    db.add_ingredient(recipe_id, "tomato sauce", 100, "g")
    db.add_ingredient(recipe_id, "mozzarella di bufala", 125, "g")
    db.add_ingredient(recipe_id, "fresh basil leaves", 8, "pcs")
    db.add_ingredient(recipe_id, "extra virgin olive oil", 15, "ml")
    db.add_ingredient(recipe_id, "salt", 5, "g")
    db.add_step(recipe_id, 1, "Preheat oven to 250°C (480°F)")
    db.add_step(recipe_id, 2, "Roll out pizza dough to 30cm diameter")
    db.add_step(recipe_id, 3, "Spread tomato sauce evenly over dough")
    db.add_step(recipe_id, 4, "Tear mozzarella and distribute over sauce")
    db.add_step(recipe_id, 5, "Season with salt and drizzle with olive oil")
    db.add_step(recipe_id, 6, "Bake for 10-12 minutes until golden")
    db.add_step(recipe_id, 7, "Add fresh basil leaves")
    db.add_step(recipe_id, 8, "Drizzle with olive oil and serve")
    print(" Created recipe 'Margherita Pizza'")

if not db.recipe_exists(italian['id'], "Tiramisu"):
    recipe_id = db.create_recipe(italian['id'], "Tiramisu", 6, 45, "Classic Italian dessert")
    db.add_ingredient(recipe_id, "ladyfingers (savoiardi)", 300, "g")
    db.add_ingredient(recipe_id, "mascarpone cheese", 500, "g")
    db.add_ingredient(recipe_id, "eggs", 4, "pcs")
    db.add_ingredient(recipe_id, "sugar", 100, "g")
    db.add_ingredient(recipe_id, "espresso coffee", 300, "ml")
    db.add_ingredient(recipe_id, "unsweetened cocoa powder", 30, "g")
    db.add_step(recipe_id, 1, "Brew espresso and let it cool")
    db.add_step(recipe_id, 2, "Separate egg yolks from whites")
    db.add_step(recipe_id, 3, "Whisk egg yolks with sugar until pale")
    db.add_step(recipe_id, 4, "Add mascarpone and mix until smooth")
    db.add_step(recipe_id, 5, "Beat egg whites with salt until stiff")
    db.add_step(recipe_id, 6, "Fold egg whites into mascarpone mixture")
    db.add_step(recipe_id, 7, "Dip ladyfingers in coffee")
    db.add_step(recipe_id, 8, "Layer ladyfingers and cream")
    db.add_step(recipe_id, 9, "Dust with cocoa powder")
    db.add_step(recipe_id, 10, "Refrigerate for 4 hours")
    print(" Created recipe 'Tiramisu'")    

   Таблица 'collections' создана
   Таблица 'recipes' создана
   Таблица 'ingredients' создана
   Таблица 'steps' создана
 База данных создана: ../data/recipes.db


In [23]:
# Создаем французскую коллекцию
if not db.collection_exists("French Cuisine"):
    french_id = db.create_collection("French Cuisine", "Classic French recipes")
    print(f" Created collection 'French Cuisine' (ID: {french_id})")
else:
    french = db.get_collection_by_name("French Cuisine")
    french_id = french['id']

# Добавляем рецепт "Creme Brulee"
if not db.recipe_exists(french_id, "Creme Brulee"):
    recipe_id = db.create_recipe(french_id, "Creme Brulee", 4, 60, "Classic French dessert with caramelized sugar")
    
    # Ингредиенты
    db.add_ingredient(recipe_id, "heavy cream", 500, "ml")
    db.add_ingredient(recipe_id, "egg yolks", 6, "pcs")
    db.add_ingredient(recipe_id, "sugar", 100, "g")
    db.add_ingredient(recipe_id, "vanilla bean", 1, "pcs")
    db.add_ingredient(recipe_id, "brown sugar", 50, "g")
    
    # Шаги
    db.add_step(recipe_id, 1, "Preheat oven to 150°C (300°F)")
    db.add_step(recipe_id, 2, "Heat cream with vanilla until simmering")
    db.add_step(recipe_id, 3, "Whisk egg yolks with sugar until pale")
    db.add_step(recipe_id, 4, "Slowly pour hot cream into egg mixture, whisking constantly")
    db.add_step(recipe_id, 5, "Strain mixture into ramekins")
    db.add_step(recipe_id, 6, "Bake in water bath for 35-40 minutes")
    db.add_step(recipe_id, 7, "Chill for at least 2 hours")
    db.add_step(recipe_id, 8, "Sprinkle brown sugar on top and caramelize with torch")
    print(" Added 'Creme Brulee'")

# Добавляем рецепт "Croissant"
if not db.recipe_exists(french_id, "Croissant"):
    recipe_id = db.create_recipe(french_id, "Croissant", 8, 180, "Buttery, flaky French pastry")
    
    db.add_ingredient(recipe_id, "all-purpose flour", 500, "g")
    db.add_ingredient(recipe_id, "butter", 250, "g")
    db.add_ingredient(recipe_id, "milk", 250, "ml")
    db.add_ingredient(recipe_id, "sugar", 50, "g")
    db.add_ingredient(recipe_id, "salt", 10, "g")
    db.add_ingredient(recipe_id, "yeast", 10, "g")
    
    db.add_step(recipe_id, 1, "Make the dough and let it rise")
    db.add_step(recipe_id, 2, "Prepare butter block")
    db.add_step(recipe_id, 3, "Laminate the dough (3 folds)")
    db.add_step(recipe_id, 4, "Roll and cut triangles")
    db.add_step(recipe_id, 5, "Shape into crescents")
    db.add_step(recipe_id, 6, "Proof until doubled")
    db.add_step(recipe_id, 7, "Egg wash and bake at 200°C for 15-18 minutes")
    print(" Added 'Croissant'")

In [24]:
# Получаем рецепт по названию
recipe = db.get_recipe_by_name(italian['id'], "Carbonara")

if recipe:
    print(f"Найден рецепт: {recipe['name']}")
    print(f"ID: {recipe['id']}")
    print(f"Порций: {recipe['servings']}")
    print(f"Время: {recipe['prep_time']} мин")

Найден рецепт: Carbonara
ID: 1
Порций: 2
Время: 30 мин


In [26]:
#all recipes
db.show_all_recipes()


 ALL RECIPES:
  1. Carbonara
     Ingredients: 5
     Steps: 6

  10. Creme Brulee
     Ingredients: 5
     Steps: 8

  11. Croissant
     Ingredients: 6
     Steps: 7

  7. Margherita Pizza
     Ingredients: 6
     Steps: 8

  9. Tiramisu
     Ingredients: 6
     Steps: 10



In [27]:
# Получаем рецепт
recipe = db.get_recipe_by_name(italian['id'], "Carbonara")

if recipe:
    # Получаем полную информацию
    full = db.get_full_recipe(recipe['id'])
    
    print(f"\n {full['name'].upper()}")
    print(f"   {full['description']}")
    print(f"   Порций: {full['servings']} | Время: {full['prep_time']} мин")
    
    print("\n Ингредиенты:")
    for ing in full['ingredients']:
        amount_str = f"{ing['amount']} {ing['unit']}" if ing['amount'] else ""
        print(f"    {ing['name']}: {amount_str}")
    
    print("\n Приготовление:")
    for step in full['steps']:
        print(f"   {step['step_order']}. {step['description']}")


 CARBONARA
   Classic Roman pasta dish
   Порций: 2 | Время: 30 мин

 Ингредиенты:
    spaghetti: 400.0 g
    guanciale: 200.0 g
    eggs: 4.0 pcs
    pecorino cheese: 100.0 g
    black pepper: 5.0 g

 Приготовление:
   1. Bring a large pot of salted water to boil
   2. Cook spaghetti according to package instructions
   3. Fry guanciale until golden and crispy, then set aside
   4. Mix eggs with pecorino cheese and pepper
   5. Combine pasta with guanciale, then add egg mixture
   6. Stir well, serve immediately


In [28]:
# Получаем рецепт
recipe = db.get_recipe_by_name(italian['id'], "Margherita Pizza")

if recipe:
    # Получаем полную информацию
    full = db.get_full_recipe(recipe['id'])
    
    print(f"\n {full['name'].upper()}")
    print(f"   {full['description']}")
    print(f"   Порций: {full['servings']} | Время: {full['prep_time']} мин")
    
    print("\n Ингредиенты:")
    for ing in full['ingredients']:
        amount_str = f"{ing['amount']} {ing['unit']}" if ing['amount'] else ""
        print(f"    {ing['name']}: {amount_str}")
    
    print("\n Приготовление:")
    for step in full['steps']:
        print(f"   {step['step_order']}. {step['description']}")


 MARGHERITA PIZZA
   Classic Neapolitan pizza with tomato, mozzarella, and basil
   Порций: 1 | Время: 20 мин

 Ингредиенты:
    pizza dough: 250.0 g
    tomato sauce: 100.0 g
    mozzarella di bufala: 125.0 g
    fresh basil leaves: 8.0 pcs
    extra virgin olive oil: 15.0 ml
    salt: 5.0 g

 Приготовление:
   1. Preheat oven to 250°C (480°F) with pizza stone if available
   2. Roll out pizza dough to 30cm diameter, about 5mm thick
   3. Spread tomato sauce evenly over dough, leaving 2cm border
   4. Tear mozzarella into pieces and distribute over sauce
   5. Season with salt and drizzle with olive oil
   6. Bake for 10-12 minutes until crust is golden and cheese bubbles
   7. Remove from oven, add fresh basil leaves
   8. Drizzle with extra olive oil and serve immediately


In [29]:
# Получаем рецепт
recipe = db.get_recipe_by_name(italian['id'], "Tiramisu")

if recipe:
    # Получаем полную информацию
    full = db.get_full_recipe(recipe['id'])
    
    print(f"\n {full['name'].upper()}")
    print(f"   {full['description']}")
    print(f"   Порций: {full['servings']} | Время: {full['prep_time']} мин")
    
    print("\n Ингредиенты:")
    for ing in full['ingredients']:
        amount_str = f"{ing['amount']} {ing['unit']}" if ing['amount'] else ""
        print(f"    {ing['name']}: {amount_str}")
    
    print("\n Приготовление:")
    for step in full['steps']:
        print(f"   {step['step_order']}. {step['description']}")


 TIRAMISU
   Classic Italian dessert
   Порций: 6 | Время: 45 мин

 Ингредиенты:
    ladyfingers (savoiardi): 300.0 g
    mascarpone cheese: 500.0 g
    eggs: 4.0 pcs
    sugar: 100.0 g
    espresso coffee: 300.0 ml
    unsweetened cocoa powder: 30.0 g

 Приготовление:
   1. Brew espresso and let it cool
   2. Separate egg yolks from whites
   3. Whisk egg yolks with sugar until pale
   4. Add mascarpone and mix until smooth
   5. Beat egg whites with salt until stiff
   6. Fold egg whites into mascarpone mixture
   7. Dip ladyfingers in coffee
   8. Layer ladyfingers and cream
   9. Dust with cocoa powder
   10. Refrigerate for 4 hours


In [30]:
# Получаем рецепт
recipe = db.get_recipe_by_name(french['id'], "Creme Brulee")

if recipe:
    # Получаем полную информацию
    full = db.get_full_recipe(recipe['id'])
    
    print(f"\n {full['name'].upper()}")
    print(f"   {full['description']}")
    print(f"   Порций: {full['servings']} | Время: {full['prep_time']} мин")
    
    print("\n Ингредиенты:")
    for ing in full['ingredients']:
        amount_str = f"{ing['amount']} {ing['unit']}" if ing['amount'] else ""
        print(f"    {ing['name']}: {amount_str}")
    
    print("\n Приготовление:")
    for step in full['steps']:
        print(f"   {step['step_order']}. {step['description']}")


 CREME BRULEE
   Classic French dessert with caramelized sugar
   Порций: 4 | Время: 60 мин

 Ингредиенты:
    heavy cream: 500.0 ml
    egg yolks: 6.0 pcs
    sugar: 100.0 g
    vanilla bean: 1.0 pcs
    brown sugar: 50.0 g

 Приготовление:
   1. Preheat oven to 150°C (300°F)
   2. Heat cream with vanilla until simmering
   3. Whisk egg yolks with sugar until pale
   4. Slowly pour hot cream into egg mixture, whisking constantly
   5. Strain mixture into ramekins
   6. Bake in water bath for 35-40 minutes
   7. Chill for at least 2 hours
   8. Sprinkle brown sugar on top and caramelize with torch


In [31]:
# Получаем рецепт
recipe = db.get_recipe_by_name(french['id'], "Croissant")

if recipe:
    # Получаем полную информацию
    full = db.get_full_recipe(recipe['id'])
    
    print(f"\n {full['name'].upper()}")
    print(f"   {full['description']}")
    print(f"   Порций: {full['servings']} | Время: {full['prep_time']} мин")
    
    print("\n Ингредиенты:")
    for ing in full['ingredients']:
        amount_str = f"{ing['amount']} {ing['unit']}" if ing['amount'] else ""
        print(f"    {ing['name']}: {amount_str}")
    
    print("\n Приготовление:")
    for step in full['steps']:
        print(f"   {step['step_order']}. {step['description']}")


 CROISSANT
   Buttery, flaky French pastry
   Порций: 8 | Время: 180 мин

 Ингредиенты:
    all-purpose flour: 500.0 g
    butter: 250.0 g
    milk: 250.0 ml
    sugar: 50.0 g
    salt: 10.0 g
    yeast: 10.0 g

 Приготовление:
   1. Make the dough and let it rise
   2. Prepare butter block
   3. Laminate the dough (3 folds)
   4. Roll and cut triangles
   5. Shape into crescents
   6. Proof until doubled
   7. Egg wash and bake at 200°C for 15-18 minutes


In [32]:
# Удаляем все рецепты Margherita Pizza, у которых нет ингредиентов
with sqlite3.connect(db.db_path) as conn:
    # Удаляем рецепты без ингредиентов
    conn.execute("""
        DELETE FROM recipes 
        WHERE name = 'Tiramisu' 
        AND id NOT IN (
            SELECT DISTINCT recipe_id 
            FROM ingredients 
            WHERE recipe_id IN (SELECT id FROM recipes WHERE name = 'Tiramisu')
        )
    """)
    conn.commit()
    print(" Удалены пустые рецепты Tiramisu")

 Удалены пустые рецепты Tiramisu


In [48]:
# Добавляем Tiramisu заново
italian = db.get_collection_by_name("Italian Cuisine")

if not db.recipe_exists(italian['id'], "Tiramisu"):
    recipe_id = db.create_recipe(italian['id'], "Tiramisu", 6, 45, "Classic Italian dessert")
    db.add_ingredient(recipe_id, "ladyfingers (savoiardi)", 300, "g")
    db.add_ingredient(recipe_id, "mascarpone cheese", 500, "g")
    db.add_ingredient(recipe_id, "eggs", 4, "pcs")
    db.add_ingredient(recipe_id, "sugar", 100, "g")
    db.add_ingredient(recipe_id, "espresso coffee", 300, "ml")
    db.add_ingredient(recipe_id, "unsweetened cocoa powder", 30, "g")
    db.add_step(recipe_id, 1, "Brew espresso and let it cool")
    db.add_step(recipe_id, 2, "Separate egg yolks from whites")
    db.add_step(recipe_id, 3, "Whisk egg yolks with sugar until pale")
    db.add_step(recipe_id, 4, "Add mascarpone and mix until smooth")
    db.add_step(recipe_id, 5, "Beat egg whites with salt until stiff")
    db.add_step(recipe_id, 6, "Fold egg whites into mascarpone mixture")
    db.add_step(recipe_id, 7, "Dip ladyfingers in coffee")
    db.add_step(recipe_id, 8, "Layer ladyfingers and cream")
    db.add_step(recipe_id, 9, "Dust with cocoa powder")
    db.add_step(recipe_id, 10, "Refrigerate for 4 hours")
    print(" Tiramisu добавлен!")
else:
    print("Tiramisu уже есть")

 Рецепт 'Tiramisu' создан (ID: 12)
   Ингредиент 'ladyfingers (savoiardi)' добавлен
   Ингредиент 'mascarpone cheese' добавлен
   Ингредиент 'eggs' добавлен
   Ингредиент 'sugar' добавлен
   Ингредиент 'espresso coffee' добавлен
   Ингредиент 'unsweetened cocoa powder' добавлен
   Шаг 1 добавлен
   Шаг 2 добавлен
   Шаг 3 добавлен
   Шаг 4 добавлен
   Шаг 5 добавлен
   Шаг 6 добавлен
   Шаг 7 добавлен
   Шаг 8 добавлен
   Шаг 9 добавлен
   Шаг 10 добавлен
 Tiramisu добавлен!


In [49]:
italian = db.get_collection_by_name("Italian Cuisine")
recipes = db.list_recipes(italian['id'])

print(f"Рецепты в '{italian['name']}':")
for r in recipes:
    print(f"  - {r['name']}")

Рецепты в 'Italian Cuisine':
  - Carbonara
  - Margherita Pizza
  - Tiramisu


In [33]:
from typing import TypedDict, Optional, List, Annotated
from langchain_core.messages import BaseMessage
import operator

class AgentState(TypedDict):
    """Состояние агента."""
    messages: Annotated[List[BaseMessage], operator.add]  # история диалога
    active_collection_id: Optional[int]  # ID активной коллекции
    active_recipe_id: Optional[int]      # ID активного рецепта

In [44]:
from langchain_core.tools import tool

_db_instance = None
_active_collection_id = None
_active_recipe_id = None

def set_db(db):
    """Установить экземпляр базы данных."""
    global _db_instance
    _db_instance = db

def get_db():
    """Получить экземпляр базы данных."""
    global _db_instance
    if _db_instance is None:
        raise RuntimeError("Database not initialized. Call set_db() first.")
    return _db_instance

# 3. Потом создаем базу
db = RecipeDB()

# 4. Потом устанавливаем БД для инструментов
set_db(db)

@tool
def create_collection(name: str, description: str = "") -> str:
    """Создать новую коллекцию рецептов."""
    db = get_db()
    
    if db.collection_exists(name):
        return f" Коллекция '{name}' уже существует"
    
    collection_id = db.create_collection(name, description)
    # Возвращаем ID, чтобы граф мог обновить состояние
    return f"COLLECTION_CREATED:{collection_id}|{name}"


@tool
def list_collections() -> str:
    """Показать все коллекции."""
    db = get_db()
    collections = db.list_collections()
    
    if not collections:
        return " Нет коллекций"
    
    result = " Коллекции:\n"
    for col in collections:
        result += f"    {col['name']}\n"
    return result

@tool
def list_recipes() -> str:
    """Показать рецепты в активной коллекции."""
    global _active_collection_id
    db = get_db()
    
    if _active_collection_id is None:
        return " Нет активной коллекции"
    
    recipes = db.list_recipes(_active_collection_id)
    collection = db.get_collection(_active_collection_id)
    
    if not recipes:
        return f" Нет рецептов в коллекции '{collection['name']}'"
    
    result = f" Рецепты в '{collection['name']}':\n"
    for i, r in enumerate(recipes, 1):
        result += f"   {i}. {r['name']}\n"
    return result

@tool
def set_active_collection(name: str) -> str:
    """Выбрать активную коллекцию по названию."""
    global _active_collection_id
    db = get_db()
    collection = db.get_collection_by_name(name)
    if not collection:
        return f" Коллекция '{name}' не найдена"
    
    _active_collection_id = collection['id']
    return f" Активная коллекция: '{name}'"


@tool
def create_recipe(name: str, servings: int, prep_time: int, description: str = "") -> str:
    """Создать новый рецепт."""
    global _active_collection_id, _active_recipe_id
    db = get_db()
    
    if _active_collection_id is None:
        return " Нет активной коллекции"
    
    if db.recipe_exists(_active_collection_id, name):
        return f" Рецепт '{name}' уже существует"
    
    recipe_id = db.create_recipe(_active_collection_id, name, servings, prep_time, description)
    _active_recipe_id = recipe_id
    return f" Рецепт '{name}' создан"



@tool
def set_active_recipe(name: str) -> str:
    """Выбрать активный рецепт по названию."""
    global _active_collection_id, _active_recipe_id
    db = get_db()
    
    if _active_collection_id is None:
        return " Нет активной коллекции"
    
    recipe = db.get_recipe_by_name(_active_collection_id, name)
    if not recipe:
        return f" Рецепт '{name}' не найден"
    
    _active_recipe_id = recipe['id']
    return f" Активный рецепт: '{name}'"


@tool
def view_recipe() -> str:
    """Показать полный рецепт."""
    global _active_recipe_id 
    db = get_db()
    
    if _active_recipe_id is None:
        return " Нет активного рецепта"
    
    full = db.get_full_recipe(_active_recipe_id)
    result = f"\n {full['name'].upper()}\n"
    result += f"   Порций: {full['servings']} | Время: {full['prep_time']} мин\n"
    result += "\n Ингредиенты:\n"
    for ing in full['ingredients']:
        result += f"    {ing['name']}: {ing['amount']} {ing['unit']}\n"
    result += "\n Приготовление:\n"
    for step in full['steps']:
        result += f"   {step['step_order']}. {step['description']}\n"
    return result


@tool
def add_ingredient(name: str, amount: float, unit: str) -> str:
    """Добавить ингредиент в активный рецепт."""
    global _active_recipe_id
    db = get_db()
    
    if _active_recipe_id is None:
        return " Нет активного рецепта"
    
    db.add_ingredient(_active_recipe_id, name, amount, unit)
    return f" Ингредиент '{name} ({amount} {unit})' добавлен"


@tool
def add_step(description: str) -> str:
    """Добавить шаг в активный рецепт."""
    global _active_recipe_id
    db = get_db()
    
    if _active_recipe_id is None:
        return " Нет активного рецепта"
    
    steps = db.get_steps(_active_recipe_id)
    step_order = len(steps) + 1
    db.add_step(_active_recipe_id, step_order, description)
    return f" Шаг {step_order} добавлен"


@tool
def edit_step(step_number: int, new_description: str) -> str:
    """Редактировать шаг."""
    global _active_recipe_id
    db = get_db()
    
    if _active_recipe_id is None:
        return " Нет активного рецепта"
    
    db.update_step(_active_recipe_id, step_number, new_description)
    return f" Шаг {step_number} обновлен"


# Список всех инструментов
tools = [
    create_collection,
    list_collections,
    set_active_collection,
    list_recipes,
    create_recipe,
    set_active_recipe,
    view_recipe,
    add_ingredient,
    add_step,
    edit_step
]

   Таблица 'collections' создана
   Таблица 'recipes' создана
   Таблица 'ingredients' создана
   Таблица 'steps' создана
 База данных создана: ../data/recipes.db


In [45]:
import sqlite3
import os
from typing import TypedDict, Optional, List, Annotated
import operator
from langchain_core.tools import tool
from langchain_core.messages import AIMessage, BaseMessage
from langchain_openai import ChatOpenAI  
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from dotenv import load_dotenv
from langgraph.prebuilt import ToolNode
from langchain_core.messages import ToolMessage

load_dotenv()

def execute_tools(state: AgentState) -> AgentState:
    """Выполняет инструменты и обновляет состояние."""
    messages = state["messages"]
    last_message = messages[-1]
    
    if not hasattr(last_message, "tool_calls") or not last_message.tool_calls:
        return state
    
    new_state = state.copy()
    
    for tool_call in last_message.tool_calls:
        tool_name = tool_call["name"]
        args = tool_call["args"]
        
        # Находим и вызываем инструмент
        tool_func = next((t for t in tools if t.name == tool_name), None)
        if not tool_func:
            continue
        
        result = tool_func.invoke(args)
        
        # Обрабатываем результат и обновляем состояние
        if result.startswith("COLLECTION_CREATED:"):
            # Обновляем активную коллекцию
            _, data = result.split(":", 1)
            collection_id, name = data.split("|", 1)
            new_state["active_collection_id"] = int(collection_id)
            result = f" Коллекция '{name}' создана (ID: {collection_id})"
        
        elif result.startswith("ACTIVE_COLLECTION:"):
            _, data = result.split(":", 1)
            collection_id, name = data.split("|", 1)
            new_state["active_collection_id"] = int(collection_id)
            result = f" Активная коллекция: '{name}'"
        
        elif result.startswith("CREATE_RECIPE:"):
            if new_state.get("active_collection_id") is None:
                result = " Нет активной коллекции"
            else:
                _, data = result.split(":", 1)
                name, servings, prep_time, description = data.split("|", 3)
                db = get_db()
                recipe_id = db.create_recipe(
                    new_state["active_collection_id"],
                    name, int(servings), int(prep_time), description
                )
                new_state["active_recipe_id"] = recipe_id
                result = f" Рецепт '{name}' создан (ID: {recipe_id})"
        
        elif result.startswith("LIST_RECIPES"):
            if new_state.get("active_collection_id") is None:
                result = " Нет активной коллекции"
            else:
                db = get_db()
                recipes = db.list_recipes(new_state["active_collection_id"])
                collection = db.get_collection(new_state["active_collection_id"])
                if not recipes:
                    result = f" Нет рецептов в '{collection['name']}'"
                else:
                    result = f" Рецепты в '{collection['name']}':\n"
                    for i, r in enumerate(recipes, 1):
                        result += f"   {i}. {r['name']}\n"
        
        elif result.startswith("SET_ACTIVE_RECIPE:"):
            if new_state.get("active_collection_id") is None:
                result = " Нет активной коллекции"
            else:
                _, name = result.split(":", 1)
                db = get_db()
                recipe = db.get_recipe_by_name(new_state["active_collection_id"], name)
                if not recipe:
                    result = f" Рецепт '{name}' не найден"
                else:
                    new_state["active_recipe_id"] = recipe['id']
                    result = f" Активный рецепт: '{name}'"
        
        elif result.startswith("VIEW_RECIPE"):
            if new_state.get("active_recipe_id") is None:
                result = " Нет активного рецепта"
            else:
                db = get_db()
                full = db.get_full_recipe(new_state["active_recipe_id"])
                result = f"\n {full['name'].upper()}\n"
                result += f"   Порций: {full['servings']} | Время: {full['prep_time']} мин\n"
                result += "\n Ингредиенты:\n"
                for ing in full['ingredients']:
                    result += f"    {ing['name']}: {ing['amount']} {ing['unit']}\n"
                result += "\n Приготовление:\n"
                for step in full['steps']:
                    result += f"   {step['step_order']}. {step['description']}\n"
        
        elif result.startswith("ADD_INGREDIENT:"):
            if new_state.get("active_recipe_id") is None:
                result = " Нет активного рецепта"
            else:
                _, data = result.split(":", 1)
                name, amount, unit = data.split("|", 2)
                db = get_db()
                db.add_ingredient(new_state["active_recipe_id"], name, float(amount), unit)
                result = f" Ингредиент '{name} ({amount} {unit})' добавлен"
        
        elif result.startswith("ADD_STEP:"):
            if new_state.get("active_recipe_id") is None:
                result = " Нет активного рецепта"
            else:
                _, description = result.split(":", 1)
                db = get_db()
                steps = db.get_steps(new_state["active_recipe_id"])
                step_order = len(steps) + 1
                db.add_step(new_state["active_recipe_id"], step_order, description)
                result = f" Шаг {step_order} добавлен"
        
        elif result.startswith("EDIT_STEP:"):
            if new_state.get("active_recipe_id") is None:
                result = " Нет активного рецепта"
            else:
                _, data = result.split(":", 1)
                step_number, new_description = data.split("|", 1)
                db = get_db()
                db.update_step(new_state["active_recipe_id"], int(step_number), new_description)
                result = f" Шаг {step_number} обновлен"
        
        # Добавляем результат в сообщения
        new_state["messages"].append(AIMessage(content=result))
    
    return new_state

In [46]:
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver

def call_llm(state: AgentState) -> AgentState:
    """Вызывает LLM с инструментами."""
    llm = ChatOpenAI(
        model="deepseek-chat",
        openai_api_key=os.getenv("DEEPSEEK_API_KEY"),
        base_url="https://api.deepseek.com",
        temperature=0
    )
    
    llm_with_tools = llm.bind_tools(tools)
    response = llm_with_tools.invoke(state["messages"])
    
    return {"messages": [response]}

# Создаем узел для инструментов с помощью ToolNode
tool_node = ToolNode(tools)

# Строим граф
workflow = StateGraph(AgentState)

# Добавляем узлы
workflow.add_node("agent", call_llm)
workflow.add_node("tools", tool_node)

# Добавляем ребра
workflow.set_entry_point("agent")
workflow.add_conditional_edges(
    "agent",
    lambda state: "tools" if state["messages"][-1].tool_calls else "end",
    {"tools": "tools", "end": END}
)
workflow.add_edge("tools", "agent")

checkpointer = MemorySaver()
app = workflow.compile(checkpointer=checkpointer)

In [99]:
def run_cli():
    """Запуск CLI для общения с агентом."""
    print(" Recipe Agent готов к работе!")
    print("Команды: /exit для выхода\n")
    
    config = {"configurable": {"thread_id": "1"}}
    
    while True:
        user_input = input(" Вы: ")
        if user_input.lower() in ["/exit", "exit", "quit"]:
            print("До свидания!")
            break
        
        # Отправляем запрос агенту
        for event in app.stream(
            {"messages": [("user", user_input)]},
            config=config,
            stream_mode="values"
        ):
            if "messages" in event:
                last_message = event["messages"][-1]
                if hasattr(last_message, "content") and last_message.content:
                    print(f" Агент: {last_message.content}")
                elif hasattr(last_message, "tool_calls"):
                    print(f" Вызов инструмента: {last_message.tool_calls[0]['name']}")

if __name__ == "__main__":
    run_cli()

 Recipe Agent готов к работе!
Команды: /exit для выхода



 Вы:  какие рецепты есть из французской кухни?


 Агент: Я помогу вам найти рецепты французской кухни. Сначала мне нужно посмотреть, какие коллекции рецептов у вас есть, а затем проверить рецепты в них.
 Агент:  Коллекции:
    French Cuisine
    Italian Cuisine
    Test Cuisine

 Агент: Отлично! У вас есть коллекция "French Cuisine". Теперь давайте выберем ее как активную и посмотрим, какие рецепты в ней содержатся.
 Агент: ACTIVE_COLLECTION:2|French Cuisine
 Агент: Теперь посмотрим рецепты в этой коллекции:
 Агент: LIST_RECIPES
 Агент: Похоже, в коллекции "French Cuisine" пока нет рецептов. Возможно, вы еще не добавили рецепты в эту коллекцию, или они находятся в других коллекциях.

Хотите, чтобы я:
1. Создал для вас несколько классических французских рецептов?
2. Или проверил другие коллекции на наличие французских рецептов?

Если вы хотите добавить французские рецепты, я могу помочь вам создать такие классические блюда как:
- Круассаны
- Рататуй
- Бёф бургиньон
- Киш Лорен
- Суп луковый по-французски
- Крем-брюле

Какие рецепты ва

 Вы:  проверь итальянскую коллекцию


 Агент: Хорошо, проверю итальянскую коллекцию.
 Агент: ACTIVE_COLLECTION:1|Italian Cuisine
 Агент: Теперь посмотрим рецепты в итальянской коллекции:
 Агент: LIST_RECIPES
 Агент: В итальянской коллекции также нет рецептов. Давайте проверим последнюю коллекцию "Test Cuisine":
 Агент: ACTIVE_COLLECTION:3|Test Cuisine
 Вызов инструмента: list_recipes
 Агент: LIST_RECIPES
 Агент: Во всех трех коллекциях пока нет рецептов. Похоже, вы создали коллекции, но еще не добавили в них рецепты.

Если вы хотите добавить французские рецепты, я могу помочь вам создать их в коллекции "French Cuisine". Например, мы можем начать с классических французских блюд:

1. **Бёф бургиньон** (тушеная говядина в красном вине)
2. **Рататуй** (овощное рагу из Прованса)
3. **Киш Лорен** (пирог с беконом и сыром)
4. **Круассаны** (слоеные булочки)
5. **Крем-брюле** (десерт с карамельной корочкой)

Какой рецепт вы хотели бы создать первым?


...

 Агент: Отлично! Создам рецепт Крем-брюле во французской коллекции. Сначала вернемся к французской коллекции:
 Агент: ACTIVE_COLLECTION:2|French Cuisine
 Агент: Теперь создам рецепт Крем-брюле. Для этого мне нужно знать:
- Количество порций
- Время приготовления (в минутах)

Сколько порций Крем-брюле вы хотите приготовить? Обычно это 4-6 порций. И сколько времени планируете на приготовление?


 Вы:  4 порции, 30 минут


 Агент: Отлично! Создаю рецепт Крем-брюле на 4 порции с временем приготовления 30 минут.
 Агент: CREATE_RECIPE:Крем-брюле|4|30|Классический французский десерт с хрустящей карамельной корочкой и нежным ванильным кремом
 Агент: Отлично! Рецепт Крем-брюле создан. Теперь давайте добавим ингредиенты. Для классического Крем-брюле на 4 порции обычно нужны:

1. Желтки яичные
2. Сахар
3. Сливки
4. Ваниль
5. Сахар для карамелизации

Добавляю ингредиенты:
 Агент: ADD_INGREDIENT:Желтки яичные|4.0|шт
 Вызов инструмента: add_ingredient
 Агент: ADD_INGREDIENT:Сахар|50.0|г
 Вызов инструмента: add_ingredient
 Агент: ADD_INGREDIENT:Сливки 33%|250.0|мл
 Вызов инструмента: add_ingredient
 Агент: ADD_INGREDIENT:Ванильный экстракт|1.0|ч.л.
 Вызов инструмента: add_ingredient
 Агент: ADD_INGREDIENT:Сахар для карамелизации|4.0|ст.л.
 Агент: Теперь добавлю шаги приготовления:
 Агент: ADD_STEP:Разогреть духовку до 150°C. В миске взбить желтки с сахаром до светлой пены
 Вызов инструмента: add_step
 Агент: ADD_STE

 Вы:  exit


До свидания!


In [73]:
# tools_dict.py
# Словарь, который будет использоваться для вызова инструментов по имени

#tools_dict = {
 #   "create_collection": create_collection,
 #   "list_collections": list_collections,
 #   "set_active_collection": set_active_collection,
 #   "set_active_recipe": set_active_recipe,
    # Эти будут вызываться через обертку с контекстом
}

In [47]:
def run_cli():
    """Запуск CLI для общения с агентом."""
    print(" Recipe Agent готов к работе!")
    print("Команды: /exit для выхода\n")
    
    config = {"configurable": {"thread_id": "1"}}
    
    while True:
        user_input = input(" Вы: ")
        if user_input.lower() in ["/exit", "exit", "quit"]:
            print("До свидания!")
            break
        
        # Отправляем запрос агенту
        for event in app.stream(
            {"messages": [("user", user_input)]},
            config=config,
            stream_mode="values"
        ):
            if "messages" in event:
                last_message = event["messages"][-1]
                if hasattr(last_message, "content") and last_message.content:
                    print(f" Агент: {last_message.content}")
                elif hasattr(last_message, "tool_calls"):
                    print(f" Вызов инструмента: {last_message.tool_calls[0]['name']}")

if __name__ == "__main__":
    run_cli()

 Recipe Agent готов к работе!
Команды: /exit для выхода



 Вы:  покажи итальянские рецепты


 Агент: Я понимаю, что вы хотите увидеть итальянские рецепты, но у меня нет возможности напрямую искать рецепты по категориям или кухням. 

Однако я могу помочь вам организовать ваши рецепты в коллекции. Давайте сначала посмотрим, какие коллекции у вас уже есть:
 Агент:  Коллекции:
    French Cuisine
    Italian Cuisine
    Test Cuisine

 Агент: Отлично! У вас уже есть коллекция "Italian Cuisine". Давайте сделаем ее активной и посмотрим, какие рецепты в ней содержатся:
 Агент:  Активная коллекция: 'Italian Cuisine'
 Агент: Теперь посмотрим рецепты в этой коллекции:
 Агент:  Рецепты в 'Italian Cuisine':
   1. Carbonara
   2. Margherita Pizza

 Агент: В вашей коллекции "Italian Cuisine" есть два рецепта:
1. Carbonara
2. Margherita Pizza

Если вы хотите посмотреть подробности какого-либо из этих рецептов, просто скажите, какой именно вас интересует, и я сделаю его активным, чтобы показать полную информацию.


 Вы:  а куда третий рецепт делся?


 Агент: Это хороший вопрос! Давайте проверим, что происходит. Я вижу, что в коллекции "Italian Cuisine" отображаются только 2 рецепта, но вы ожидаете, что там должен быть третий.

Давайте проверим все коллекции, чтобы убедиться, что рецепт не находится в другой коллекции:
 Агент:  Коллекции:
    French Cuisine
    Italian Cuisine
    Test Cuisine

 Агент: Проверим рецепты в других коллекциях. Сначала в "French Cuisine":
 Агент:  Активная коллекция: 'French Cuisine'
 Вызов инструмента: list_recipes
 Агент:  Рецепты в 'French Cuisine':
   1. Creme Brulee
   2. Croissant

 Агент: Теперь проверим "Test Cuisine":
 Агент:  Активная коллекция: 'Test Cuisine'
 Вызов инструмента: list_recipes
 Агент:  Нет рецептов в коллекции 'Test Cuisine'
 Агент: Похоже, что в системе действительно только 4 рецепта:
- В "Italian Cuisine": Carbonara и Margherita Pizza (2 рецепта)
- В "French Cuisine": Creme Brulee и Croissant (2 рецепта)

Возможно, третий рецепт был удален из системы, или вы создавали его в др

 Вы:  exit


До свидания!


In [84]:
# Инициализируем
#from src.recipe_agent.tools import set_db, create_collection, list_collections

#set_db(db)

# Проверяем создание коллекции
#result = create_collection.invoke({"name": "Test Cuisine", "description": "Test collection"})
#print(result)

# Проверяем список коллекций
#result = list_collections.invoke({})
#print(result)

Collection 'Test Cuisine' already exists.
 Collections:
   French Cuisine - French classics
   Italian Cuisine - Classic Italian recipes
   Test Cuisine - Test collection



In [85]:
#tools = [
 #   create_collection,
#    list_collections,
 #   set_active_collection,
#]

#print(" Доступные инструменты для LLM:")
#for tool in tools:
#    print(f"   - {tool.name}: {tool.description}")

 Доступные инструменты для LLM:
   - create_collection: Create a new recipe collection.

Args:
    name: Name of the collection (e.g., "Italian Cuisine")
    description: Optional description of the collection
   - list_collections: List all recipe collections.
   - set_active_collection: Set the active collection by name.

Args:
    name: Name of the collection


In [100]:
# Проверка, что агент видит рецепты
collections = db.list_collections()
print("Коллекции в базе агента:")
for col in collections:
    print(f"  - {col['name']}")

for col in collections:
    recipes = db.list_recipes(col['id'])
    print(f"\nРецепты в {col['name']}:")
    for recipe in recipes:
        print(f"  - {recipe['name']}")

Коллекции в базе агента:
  - French Cuisine
  - Italian Cuisine
  - Test Cuisine

Рецепты в French Cuisine:
  - Creme Brulee
  - Croissant

Рецепты в Italian Cuisine:
  - Carbonara
  - Margherita Pizza
  - Tiramisu

Рецепты в Test Cuisine:
